# AI-Driven Logistics and Shipment Analytics
**Prepared by:** Vaibhavi Kadam

This notebook demonstrates shipment data cleaning, EDA, visualizations, and a baseline model to estimate delivery delay. It uses `logistics_data.csv` if supplied; otherwise it creates clearly labeled synthetic demo data. Synthetic results are not real-world findings.

Suggested dataset: https://www.kaggle.com/datasets/shashwatwork/dataco-smart-supply-chain-for-big-data-analysis


In [ ]:
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
np.random.seed(42)


In [ ]:
DATA_FILE='logistics_data.csv'
USING_DEMO_DATA = not os.path.exists(DATA_FILE)
if not USING_DEMO_DATA:
    df=pd.read_csv(DATA_FILE)
else:
    n=800
    mode=np.random.choice(['Air','Truck','Ocean','Air Charter'],n,p=[.42,.35,.18,.05])
    distance=np.random.randint(100,6000,n); weight=np.random.randint(10,15000,n)
    product=np.random.choice(['ARV','HRDT','ANTM','ACT'],n,p=[.68,.22,.07,.03])
    country=np.random.choice(['India','Kenya','Nigeria','South Africa','Vietnam','Uganda'],n)
    effect=pd.Series(mode).map({'Air':1,'Truck':2.5,'Ocean':5,'Air Charter':.5}).to_numpy()
    delay=np.round(distance/1500+weight/7000+effect+np.random.normal(0,1.8,n)).astype(int)
    df=pd.DataFrame({'Shipment Mode':mode,'Country':country,'Product Group':product,'Distance_km':distance,'Weight_kg':weight,'Order_Value_USD':np.random.gamma(2,2500,n).round(2),'Delivery_Delay_Days':delay})
print('SYNTHETIC DEMO DATA — not real-world findings' if USING_DEMO_DATA else 'Loaded logistics_data.csv')
print('Shape:',df.shape); display(df.head())


In [ ]:
# Quality checks and basic cleaning
print('Missing values:'); display(df.isna().sum().to_frame('missing'))
print('Duplicate rows:',df.duplicated().sum())
df=df.drop_duplicates().copy()
for c in df.select_dtypes(include='object').columns: df[c]=df[c].astype('string').str.strip()
display(df.describe(include='all').T)


In [ ]:
# EDA
target='Delivery_Delay_Days'
if target not in df.columns: raise ValueError('Rename your dataset target column to Delivery_Delay_Days before modelling.')
plt.figure(figsize=(8,4)); df[target].dropna().plot(kind='hist',bins=25,edgecolor='black'); plt.title('Delivery Delay Distribution'); plt.xlabel('Delay (days)'); plt.tight_layout(); plt.show()
for c in ['Shipment Mode','Product Group']:
    if c in df.columns:
        plt.figure(figsize=(8,4)); df[c].value_counts().plot(kind='bar',edgecolor='black'); plt.title('Shipment Count by '+c); plt.tight_layout(); plt.show()
if 'Shipment Mode' in df.columns: display(df.groupby('Shipment Mode')[target].agg(['count','mean','median']))


In [ ]:
# Train and evaluate baseline Random Forest regressor
work=df.dropna(subset=[target]).copy(); X=work.drop(columns=[target]); y=work[target]
num=X.select_dtypes(include='number').columns.tolist(); cat=X.select_dtypes(include=['object','string','category']).columns.tolist()
pre=ColumnTransformer([('num',Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat)])
model=Pipeline([('preprocess',pre),('rf',RandomForestRegressor(n_estimators=150,random_state=42,min_samples_leaf=2))])
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42); model.fit(Xtr,ytr); pred=model.predict(Xte)
mae=mean_absolute_error(yte,pred); rmse=np.sqrt(mean_squared_error(yte,pred)); r2=r2_score(yte,pred)
print(f'MAE: {mae:.3f} days | RMSE: {rmse:.3f} days | R2: {r2:.3f}')
print('Metrics describe synthetic demo data only.' if USING_DEMO_DATA else 'Metrics describe the loaded dataset.')
plt.figure(figsize=(6,5)); plt.scatter(yte,pred,alpha=.6); plt.xlabel('Actual delay'); plt.ylabel('Predicted delay'); plt.title('Actual vs Predicted'); plt.tight_layout(); plt.show()
df.to_csv('cleaned_logistics_data.csv',index=False)
pd.DataFrame([{'rows':len(df),'columns':len(df.columns),'synthetic_demo_data':USING_DEMO_DATA,'MAE_days':mae,'RMSE_days':rmse,'R2':r2}]).to_csv('model_evaluation_summary.csv',index=False)


## Conclusion
This is a starter project template. Replace the synthetic demo data with a suitable real dataset, map its fields correctly, run all cells, and use the actual outputs in your final report. Do not present demo metrics as real findings.
